## Mid-project: EDA and baseline models

**Dataset:** Government of Canada light-duty vehicle fuel consumption ratings (CO2 emissions in g/km).  
**Goal:** Regress CO2 from vehicle attributes using a reproducible sklearn pipeline (scaling + one-hot encoding) and compare linear, k-NN, and SVM baselines.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

sns.set_theme(style="whitegrid")

# Resolve data path whether the kernel cwd is repo root or notebooks/
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / "data").is_dir() else _cwd.parent
DATA_PATH = PROJECT_ROOT / "data" / "CO2 Emissions_Canada.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

target = "CO2 Emissions(g/km)"
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())

numeric_features = ["Engine Size(L)", "Cylinders", "Fuel Consumption Comb (L/100 km)"]
categorical_features = ["Fuel Type", "Transmission"]

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df[target], kde=True, ax=ax)
ax.set_title("Distribution of CO2 emissions (g/km)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_co2_distribution.png", dpi=150)
plt.show()

eda = df[numeric_features + [target]]
plt.figure(figsize=(8, 6))
sns.heatmap(eda.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric feature correlation (including target)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_correlation_numeric.png", dpi=150)
plt.show()

numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
try:
    _ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    _ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
categorical_transformer = Pipeline(steps=[("onehot", _ohe)])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X = df.drop(target, axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression (baseline)": LinearRegression(),
    "k-NN (k=5)": KNeighborsRegressor(n_neighbors=5),
    "SVR (RBF, C=100)": SVR(kernel="rbf", C=100),
}

print("--- Mid-project preliminary results (hold-out test, 20%) ---")
rows = []
for name, model in models.items():
    clf = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rows.append({"model": name, "r2": r2, "mae_g_per_km": mae})
    print(f"{name} -> R2: {r2:.4f}, MAE: {mae:.2f}")

results = pd.DataFrame(rows)
fig, ax1 = plt.subplots(figsize=(8, 4))
x = np.arange(len(results))
ax1.bar(x - 0.2, results["r2"], 0.4, label="R2", color="steelblue")
ax2 = ax1.twinx()
ax2.bar(x + 0.2, results["mae_g_per_km"], 0.4, label="MAE", color="coral")
ax1.set_xticks(x)
ax1.set_xticklabels(results["model"], rotation=15, ha="right")
ax1.set_ylabel("R2")
ax2.set_ylabel("MAE (g/km)")
ax1.set_ylim(0, 1.05)
ax1.set_title("Model comparison (notebook run)")
fig.legend(loc="upper right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_model_comparison_notebook.png", dpi=150)
plt.show()

best = results.sort_values("mae_g_per_km").iloc[0]["model"]
clf_best = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", models[best]),
    ]
)
clf_best.fit(X_train, y_train)
pred = clf_best.predict(X_test)
plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.35, s=12)
lo, hi = float(y_test.min()), float(y_test.max())
plt.plot([lo, hi], [lo, hi], "r--", lw=2)
plt.xlabel("Actual CO2 (g/km)")
plt.ylabel("Predicted CO2 (g/km)")
plt.title(f"Actual vs predicted — {best}")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_actual_vs_predicted_best.png", dpi=150)
plt.show()

<>:12: SyntaxWarning: invalid escape sequence '\R'
<>:12: SyntaxWarning: invalid escape sequence '\R'
C:\Users\arcot\AppData\Local\Temp\ipykernel_23020\1336846869.py:12: SyntaxWarning: invalid escape sequence '\R'
  df = pd.read_csv('D:\Ragx\ml3\data\CO2 Emissions_Canada.csv')


--- Mid-Project Preliminary Results ---
Linear Regression (Baseline) -> R2 Score: 0.9909, MAE: 3.13
KNN Regressor -> R2 Score: 0.9852, MAE: 3.94
